# Séance 5 — Transformers & Mécanismes d'Attention
### TP Pratique — Golden Collar

Ce notebook suit exactement le fil rouge du cours :
> *"l'éléphant rose a essayé de monter dans la voiture, mais il était trop ..."*

**Plan :**
1. Self-Attention from scratch (NumPy)
2. Visualisation des poids d'attention
3. Positional Encoding : Sinus/Cosinus vs RoPE
4. Multi-Head Attention
5. Exploration de modèles pré-entraînés (HuggingFace)
6. Démo : Capacités émergentes des LLMs

---
## 0. Installation & Imports

In [ ]:
# Decommenter si premiere execution
# !pip install transformers torch numpy matplotlib seaborn scikit-learn

# Nemotron-Mini-4B-Instruct est un Transformer decoder DENSE (pas besoin de mamba-ssm).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Imports OK')

---
## 1. Self-Attention From Scratch (NumPy)

### 1.1 — Rappel de la formule

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

On va implémenter ça à la main, étape par étape, sur notre phrase exemple.

In [ ]:
def softmax(x, axis=-1):
    """Softmax numériquement stable."""
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q : (seq_len, d_k)
    K : (seq_len, d_k)
    V : (seq_len, d_v)
    mask : (seq_len, seq_len) optionnel — masque causal

    Retourne : (output, attention_weights)
    """
    d_k = Q.shape[-1]

    # Étape 1 : scores d'affinité Q·K^T
    scores = Q @ K.T / np.sqrt(d_k)        # (seq_len, seq_len)

    # Étape 2 : masquage causal (optionnel)
    if mask is not None:
        scores = scores + mask              # -inf aux positions futures

    # Étape 3 : distribution de probabilité sur les tokens
    weights = softmax(scores, axis=-1)      # (seq_len, seq_len)

    # Étape 4 : combinaison pondérée des Values
    output = weights @ V                   # (seq_len, d_v)

    return output, weights


print('Fonction scaled_dot_product_attention définie.')

In [ ]:
# --- Phrase fil rouge ---

SENTENCE = "l'éléphant rose a essayé de monter dans la voiture, mais il était trop"
tokens = SENTENCE.split()  # Tokenisation très basique (en pratique : tokenizer de Hugging Face)
print(f'Tokens : {tokens}')
n = len(tokens)   # nombre de tokens (13 avec .split() sur cette phrase)
d_model = 8       # dimension d'embedding (petit pour la lisibilité)
d_k = 4           # dimension des projections Q et K
d_v = 4           # dimension des projections V

# Embeddings simulés (en pratique : issus d'une matrice E apprise)
X = np.random.randn(n, d_model)

# Matrices de projection (apprises durant l'entraînement)
W_q = np.random.randn(d_model, d_k) * 0.1
W_k = np.random.randn(d_model, d_k) * 0.1
W_v = np.random.randn(d_model, d_v) * 0.1

# Projections Q, K, V
Q = X @ W_q   # (n, d_k)
K = X @ W_k   # (n, d_k)
V = X @ W_v   # (n, d_v)

print(f'X   shape : {X.shape}   — séquence d\'entrée')
print(f'Q   shape : {Q.shape}   — queries')
print(f'K   shape : {K.shape}   — keys')
print(f'V   shape : {V.shape}   — values')

### 1.2 — Calcul et visualisation de la matrice d'attention

In [ ]:
output, attn_weights = scaled_dot_product_attention(Q, K, V)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    attn_weights,
    annot=True, fmt='.2f',
    xticklabels=tokens,
    yticklabels=tokens,
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax
)
ax.set_title('Matrice d\'attention (Self-Attention bidirectionnelle)', fontsize=13)
ax.set_xlabel('Tokens source (Keys)')
ax.set_ylabel('Tokens cible (Queries)')
plt.tight_layout()
plt.show()

print(f'\nOutput shape : {output.shape}')
print('Somme des poids par ligne (doit être ~1.0) :', attn_weights.sum(axis=1).round(4))

### 1.3 — Masque Causal (Autoregressive)

En mode décodeur (génération), chaque token ne peut regarder que les tokens *passés*.

In [ ]:
def causal_mask(seq_len):
    """Masque triangulaire supérieur : -inf pour les positions futures."""
    mask = np.triu(np.full((seq_len, seq_len), -np.inf), k=1)
    return mask


mask = causal_mask(n)

output_causal, attn_weights_causal = scaled_dot_product_attention(Q, K, V, mask=mask)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, weights, title in zip(
    axes,
    [attn_weights, attn_weights_causal],
    ['Sans masque (Encodeur)', 'Avec masque causal (Décodeur)']
):
    sns.heatmap(
        weights, annot=True, fmt='.2f',
        xticklabels=tokens, yticklabels=tokens,
        cmap='YlOrRd', linewidths=0.5,
        vmin=0, vmax=1, ax=ax
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Keys')
    ax.set_ylabel('Queries')

plt.suptitle('Impact du masquage causal', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

> **Question 1** : Dans la matrice avec masque causal, pourquoi le token `"rose"` (2ème token, index 1) ne regarde-t-il que `"l'éléphant"` et `"rose"` lui-même ? Vérifiez que la somme des poids d'attention sur les positions *futures* vaut 0 pour chaque ligne.

> **Question 2** : Que se passe-t-il si on enlève le facteur $\sqrt{d_k}$ ? Testez ci-dessous.

In [ ]:
# --- Exercice : effet du facteur de normalisation ---

def attention_no_scale(Q, K, V):
    scores = Q @ K.T    # sans /sqrt(d_k)
    weights = softmax(scores)
    return weights @ V, weights

# Avec grand d_k pour exagérer l'effet
d_big = 512
Q_big = np.random.randn(n, d_big)
K_big = np.random.randn(n, d_big)
V_big = np.random.randn(n, d_big)

_, w_scaled   = scaled_dot_product_attention(Q_big, K_big, V_big)
_, w_unscaled = attention_no_scale(Q_big, K_big, V_big)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, w, title in zip(axes, [w_scaled, w_unscaled],
                        [f'Avec √d_k (d_k={d_big})', f'Sans √d_k (d_k={d_big})']):
    im = ax.imshow(w, cmap='hot', vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks(range(n)); ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_yticks(range(n)); ax.set_yticklabels(tokens)
    plt.colorbar(im, ax=ax)

plt.suptitle('Saturation du softmax sans normalisation', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Entropie (scalé)   : {(-w_scaled * np.log(w_scaled + 1e-9)).sum(axis=1).mean():.3f}')
print(f'Entropie (non scalé): {(-w_unscaled * np.log(w_unscaled + 1e-9)).sum(axis=1).mean():.3f}')
print('→ Une entropie proche de 0 = attention saturée ("winner-takes-all"), gradient ≈ 0')

---
## 2. Positional Encoding

### 2.1 — Encodage Sinus/Cosinus (Vaswani 2017)

In [ ]:
def positional_encoding_sincos(max_len, d_model):
    """
    Encodage positionnel absolu via sin/cos (Vaswani 2017).
    PE[pos, 2k]   = sin(pos / 10000^(2k/d_model))
    PE[pos, 2k+1] = cos(pos / 10000^(2k/d_model))
    """
    PE = np.zeros((max_len, d_model))
    pos = np.arange(max_len)[:, np.newaxis]            # (max_len, 1)
    i   = np.arange(0, d_model, 2)[np.newaxis, :]      # [0, 2, 4, ...] = 2k
    angles = pos / np.power(10000, i / d_model)        # exposant 2k/d_model
    PE[:, 0::2] = np.sin(angles)
    PE[:, 1::2] = np.cos(angles)
    return PE


max_len, d_model_vis = 50, 64
PE = positional_encoding_sincos(max_len, d_model_vis)


In [ ]:
# Similarité cosinus entre positions — clé de l'encodage positionnel
norms = np.linalg.norm(PE, axis=1, keepdims=True)
PE_norm = PE / norms
sim_matrix = PE_norm @ PE_norm.T

plt.figure(figsize=(7, 5))
sns.heatmap(sim_matrix, cmap='coolwarm', vmin=-1, vmax=1,
            xticklabels=5, yticklabels=5)
plt.title('Similarité cosinus entre positions\n(positions proches = plus similaires)')
plt.xlabel('Position j')
plt.ylabel('Position i')
plt.tight_layout()
plt.show()
print('→ La diagonale (même position) = 1.0')
print('→ Les positions proches ont une similarité élevée : le réseau peut apprendre la distance relative')

### 2.2 — RoPE (Rotary Position Embedding)

RoPE encode la position via une **rotation** dans l'espace complexe, ce qui fait que le produit scalaire $q_m \cdot k_n$ ne dépend que de la **distance relative** $(m - n)$.

In [ ]:
def rope_rotation(x, position, theta=10000.0):
    """
    Applique la rotation RoPE au vecteur x à la position donnée.
    x       : (d,)  — doit avoir une dimension paire
    position: int   — indice de position dans la séquence
    """
    d = len(x)
    assert d % 2 == 0, 'd doit être pair'

    i     = np.arange(d // 2)
    freqs = 1.0 / (theta ** (2 * i / d))
    angle = position * freqs

    x_complex = x[0::2] + 1j * x[1::2]
    rotated   = x_complex * np.exp(1j * angle)

    result = np.zeros_like(x)
    result[0::2] = rotated.real
    result[1::2] = rotated.imag
    return result


# Démonstration : propriété de distance relative de RoPE
d_rope = 8
v = np.random.randn(d_rope)

# On utilise pos_m / pos_n (pas m/n) pour ne pas écraser la variable globale n
test_pairs = [(0, 1), (5, 6), (0, 3), (10, 13), (0, 10)]

print('Propriété RoPE : le produit scalaire <q_m, k_n> ne dépend que de (m - n)\n')
print(f'{"Positions (m, n)":20s}  {"Distance":10s}  {"Produit scalaire":15s}')
print('-' * 50)

for pos_m, pos_n in test_pairs:
    q_m = rope_rotation(v, pos_m)
    k_n = rope_rotation(v, pos_n)
    dot = np.dot(q_m, k_n)
    print(f'  ({pos_m:2d}, {pos_n:2d})              {pos_n - pos_m:5d}       {dot:12.6f}')

> **Observation** : Les paires (0,1), (5,6), etc. ont la **même distance = 1** et donc des produits scalaires très proches. C'est la propriété clé de RoPE.

### Exercice RoPE — Matrice de similarité relative

**Objectif** : Vérifier empiriquement que le produit scalaire RoPE ne dépend que de la **distance** $|i - j|$, pas des positions absolues.

Complétez `rope_similarity_matrix` pour calculer $M[i, j] = \langle \text{rope}(v, i),\ \text{rope}(v, j) \rangle$ et comparez la matrice obtenue à celle du sin/cos.

In [ ]:
def rope_similarity_matrix(seq_len, d=8, theta=10000.0):
    """
    Retourne M (seq_len × seq_len) où M[i, j] = dot(rope(v, i), rope(v, j)).
    Si RoPE est correct, M doit être *Toeplitz* : M[i,j] ne dépend que de (i-j).
    """
    np.random.seed(0)
    v = np.random.randn(d)

    M = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        for j in range(seq_len):
            M[i, j] = np.dot(rope_rotation(v, i), rope_rotation(v, j))
    return M


SEQ = 20
M_rope = rope_similarity_matrix(SEQ)

# Comparaison avec sin/cos
PE_cmp  = positional_encoding_sincos(SEQ, 8)
norms   = np.linalg.norm(PE_cmp, axis=1, keepdims=True)
M_sincos = (PE_cmp / norms) @ (PE_cmp / norms).T

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, M, title in zip(axes, [M_rope, M_sincos],
                         ['RoPE — dot(rope(v,i), rope(v,j))\n(doit être Toeplitz)',
                          'Sin/Cos — similarité cosinus\n(pour comparaison)']):
    im = ax.imshow(M, cmap='coolwarm', aspect='auto')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Position j')
    ax.set_ylabel('Position i')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

# Vérification : les diagonales doivent être constantes
print("Diagonale +1 (distance 1) — doit être constante :", np.round(np.diag(M_rope, k=1), 4))

---
## 3. Multi-Head Attention

Chaque tête opère dans un **sous-espace** différent et capture des relations orthogonales.

In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model, n_heads):
        assert d_model % n_heads == 0, 'd_model doit être divisible par n_heads'
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads

        scale = 0.1
        self.W_q = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_k = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_v = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_o = np.random.randn(d_model, d_model) * scale


    def forward(self, X, mask=None):
        head_outputs = []
        all_weights  = []

        for h in range(self.n_heads):
            Q_h = X @ self.W_q[h]
            K_h = X @ self.W_k[h]
            V_h = X @ self.W_v[h]
            out_h, w_h = scaled_dot_product_attention(Q_h, K_h, V_h, mask)
            head_outputs.append(out_h)
            all_weights.append(w_h)

        concat = np.concatenate(head_outputs, axis=-1)
        output = concat @ self.W_o
        return output, np.stack(all_weights)


# Utilise len(tokens) → toujours synchronisé avec la phrase courante
d_model_mha = 16
n_heads     = 4
X_mha = np.random.randn(len(tokens), d_model_mha)

mha = MultiHeadAttention(d_model_mha, n_heads)
output_mha, all_weights = mha.forward(X_mha)

print(f'Tokens        : {tokens}')
print(f'Input shape   : {X_mha.shape}   → ({len(tokens)} tokens, d_model={d_model_mha})')
print(f'Output shape  : {output_mha.shape}')
print(f'Weights shape : {all_weights.shape}  → (n_heads, seq, seq)')

In [ ]:
# Visualisation : chaque tête voit la séquence différemment
n_tok = len(tokens)
cell_size = max(0.55, 4.5 / n_tok)          # taille d'une cellule en pouces
panel_size = n_tok * cell_size               # taille d'un panneau carré

fig, axes = plt.subplots(1, n_heads, figsize=(panel_size * n_heads + 1, panel_size + 1))

for h, ax in enumerate(axes):
    sns.heatmap(
        all_weights[h], annot=True, fmt='.2f',
        xticklabels=tokens, yticklabels=tokens,
        cmap='Blues', linewidths=0.3,
        vmin=0, vmax=1, ax=ax, cbar=False,
        annot_kws={"size": max(5, 9 - n_tok // 3)},
    )
    ax.set_title(f'Tête {h + 1}\n(d_k={d_model_mha // n_heads})', fontsize=10)
    ax.tick_params(axis='x', rotation=45, labelsize=max(6, 9 - n_tok // 4))
    ax.tick_params(axis='y', rotation=0,  labelsize=max(6, 9 - n_tok // 4))

plt.suptitle(
    f'Multi-Head Attention ({n_heads} têtes) — chaque tête apprend des relations différentes',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

#### Pourquoi plusieurs têtes et non une seule ?

Si on *moyennait* toutes les têtes au lieu de les concaténer, on obtiendrait une attention floue qui perdrait la spécialisation de chaque tête. La comparaison ci-dessous l'illustre.

In [ ]:
avg_attn = all_weights.mean(axis=0)   # (seq_len, seq_len)

n_tok      = len(tokens)
cell_size  = max(0.55, 4.5 / n_tok)
panel_size = n_tok * cell_size

fig, axes = plt.subplots(1, n_heads + 1,
                         figsize=(panel_size * (n_heads + 1) + 1, panel_size + 1))

for h in range(n_heads):
    sns.heatmap(all_weights[h], ax=axes[h], cmap='Blues', vmin=0, vmax=1,
                xticklabels=tokens, yticklabels=tokens, cbar=False, annot=False)
    axes[h].set_title(f'Tête {h + 1}', fontsize=9)
    axes[h].tick_params(axis='x', rotation=45, labelsize=max(5, 8 - n_tok // 4))
    axes[h].tick_params(axis='y', rotation=0,  labelsize=max(5, 8 - n_tok // 4))

sns.heatmap(avg_attn, ax=axes[-1], cmap='Blues', vmin=0, vmax=1,
            xticklabels=tokens, yticklabels=tokens, cbar=False, annot=False)
axes[-1].set_title('Moyenne\n(perd la spécialisation)', fontsize=9, color='crimson')
axes[-1].tick_params(axis='x', rotation=45, labelsize=max(5, 8 - n_tok // 4))
axes[-1].tick_params(axis='y', rotation=0,  labelsize=max(5, 8 - n_tok // 4))

plt.suptitle('Chaque tête se spécialise — la moyenne efface cette information',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print("→ La concaténation (pas la moyenne) des têtes préserve la diversité des représentations.")
print("→ C'est la projection W_O finale qui fusionne les sous-espaces de façon apprise.")

> **Question 3** : Augmentez le nombre de têtes à 8 (et `d_model_mha` à 32). Est-ce que les têtes se différencient davantage ? Pourquoi ?

---
## 4. Bloc Transformer Complet

On dispose maintenant de toutes les briques de l'attention. Il reste deux composants essentiels pour assembler un **vrai bloc Transformer** :

| Composant | Rôle |
|---|---|
| **Feed-Forward Network (FFN)** | Transformation non-linéaire token par token — "mémoire associative" |
| **Layer Normalization + Résidu** | Stabilisation du gradient — permet d'empiler des dizaines de couches |

```
        ┌──────────────────────────────────┐
x ──►───┤  LayerNorm → MHA → + résidu (x) ├──► x'
        └──────────────────────────────────┘
x' ──►──┤  LayerNorm → FFN → + résidu (x')├──► x''
        └──────────────────────────────────┘
```

### 4.1 Feed-Forward Network (FFN)

Le FFN est appliqué **indépendamment** à chaque token après l'attention — il joue le rôle de **mémoire associative** du modèle :

$$\text{FFN}(x) = W_2 \cdot \text{GeLU}(W_1 x + b_1) + b_2$$

avec $W_1 \in \mathbb{R}^{d \times 4d}$ (expansion) et $W_2 \in \mathbb{R}^{4d \times d}$ (projection). L'expansion $\times 4$ crée un espace latent plus riche pour stocker des associations.

> **Exercice** : Implémentez `FeedForwardNetwork.forward()`.  
> Rappel : $\text{FFN}(x) = W_2 \cdot \text{GeLU}(W_1 x + b_1) + b_2$

In [ ]:
def gelu(x):
    """Approximation de GeLU : x * Φ(x) où Φ est la CDF gaussienne."""
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))


class FeedForwardNetwork:
    def __init__(self, d_model, d_ff=None):
        self.d_model = d_model
        self.d_ff = d_ff or 4 * d_model   # expansion x4 par défaut

        scale = 0.02
        self.W1 = np.random.randn(self.d_model, self.d_ff) * scale   # (d_model, d_ff)
        self.b1 = np.zeros(self.d_ff)                                 # (d_ff,)
        self.W2 = np.random.randn(self.d_ff, self.d_model) * scale   # (d_ff, d_model)
        self.b2 = np.zeros(self.d_model)                              # (d_model,)

    def forward(self, x):
        """
        x : (seq_len, d_model)
        Retourne : (seq_len, d_model)

        Rappel : FFN(x) = W2 · GeLU(W1·x + b1) + b2
        """
        # TODO 1 : Première couche linéaire + activation GeLU
        h   = gelu(x @ self.W1 + self.b1)

        # TODO 2 : Deuxième couche linéaire (projection vers d_model)
        out = h @ self.W2 + self.b2

        return out


# --- Test ---
ffn = FeedForwardNetwork(d_model=16)
out = ffn.forward(X_mha)
assert out.shape == X_mha.shape, f"Shape incorrecte : {out.shape}"
print(f'FFN output shape : {out.shape}  OK')

### 4.2 Assemblage du Bloc Transformer (Pre-LN)

Un bloc Transformer **Pre-LN** combine le MHA et le FFN avec des connexions résiduelles :

```
x'  = x  + MHA(LayerNorm(x))
x'' = x' + FFN(LayerNorm(x'))
```

La **Layer Normalization** stabilise le gradient ; les **connexions résiduelles** permettent à l'information brute de circuler sans dégradation à travers les couches profondes.

> **Exercice** : Implémentez `TransformerBlock.forward()` en utilisant `MultiHeadAttention`, `FeedForwardNetwork` et `layer_norm` ci-dessus.

In [ ]:
def layer_norm(x, eps=1e-6):
    """Layer Normalization par vecteur (indépendant de la batch)."""
    mean = x.mean(axis=-1, keepdims=True)
    std  = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)


class TransformerBlock:
    def __init__(self, d_model, n_heads):
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForwardNetwork(d_model=d_model)

    def forward(self, x, mask=None):
        """
        x : (seq_len, d_model)
        Retourne : (seq_len, d_model)

        Architecture Pre-LN (pré-normalisation) :
            x'  = x  + MHA( LayerNorm(x) )
            x'' = x' + FFN( LayerNorm(x') )
        """
        # TODO 1 : Connexion résiduelle autour du Multi-Head Attention
        # Indice : appliquer layer_norm, puis self.mha.forward(...), puis résidu
        x_prime        = x        + self.mha.forward(layer_norm(x),       mask)[0]

        # TODO 2 : Connexion résiduelle autour du Feed-Forward Network
        x_double_prime = x_prime  + self.ffn.forward(layer_norm(x_prime))

        return x_double_prime


# --- Test ---
block = TransformerBlock(d_model=16, n_heads=4)
out = block.forward(X_mha)
assert out.shape == X_mha.shape, f"Shape incorrecte : {out.shape}"
print(f'Bloc Transformer — input : {X_mha.shape} — output : {out.shape}  OK')

---
## 5. Modèles Pré-entraînés (HuggingFace)

On va utiliser un vrai modèle pré-entraîné pour visualiser l'attention **apprise** sur des millions de documents — et comparer avec notre implémentation NumPy.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# CamemBERT : encodeur BERT entraîné sur 138 Go de texte français
# Tokenizer SentencePiece / BPE optimisé pour les accents et la morphologie française
MODEL_NAME = 'camembert-base'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

print(f'Modèle chargé : {MODEL_NAME}')
print(f'Nombre de couches : {model.config.num_hidden_layers}')
print(f'Nombre de têtes   : {model.config.num_attention_heads}')
print(f'd_model           : {model.config.hidden_size}')


In [ ]:
# Tokenisation de notre phrase fil rouge
#phrase = "The elephant tried to get into the car but it was too large"
phrase = SENTENCE

inputs = tokenizer(phrase, return_tensors='pt')
tokens_list = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

print('Tokens :', tokens_list)
print('IDs    :', inputs['input_ids'][0].tolist())

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

# outputs.attentions : tuple de (n_layers,) tenseurs (batch, n_heads, seq, seq)
attentions = outputs.attentions
print(f'Nombre de couches avec attention : {len(attentions)}')
print(f'Shape par couche : {attentions[0].shape}  → (batch, n_heads, seq_len, seq_len)')

#### Visualisation des embeddings d'entrée (avant toute couche d'attention)

Avant même la première couche d'attention, les tokens sont déjà placés dans un espace de dimension 768. Une réduction PCA montre que des tokens proches sémantiquement / syntaxiquement se regroupent naturellement.

In [ ]:
from sklearn.decomposition import PCA

# Embeddings bruts CamemBERT (table de lookup, sans positional encoding ni attention)
with torch.no_grad():
    token_embs = model.embeddings.word_embeddings(inputs['input_ids'])  # (1, seq, 768)
    embs = token_embs[0].numpy()   # (seq_len, 768)

pca     = PCA(n_components=2)
embs_2d = pca.fit_transform(embs)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(embs_2d[:, 0], embs_2d[:, 1], color='steelblue', s=90, zorder=3)
for i, tok in enumerate(tokens_list):
    ax.annotate(tok, (embs_2d[i, 0], embs_2d[i, 1]),
                textcoords='offset points', xytext=(6, 4), fontsize=8)

ax.set_title(
    f'PCA des embeddings CamemBERT (avant attention)\n"{phrase}"',
    fontsize=11
)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var.)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var.)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("→ Les tokens ont déjà une géométrie sémantique AVANT toute couche d'attention.")
print("→ L'attention va ensuite contextualiser chaque vecteur en fonction du reste de la séquence.")

In [ ]:
def plot_attention_layer(attentions, tokens_list, layer_idx=0, n_heads_show=6):
    """Visualise les n_heads_show premières têtes d'une couche donnée."""
    attn = attentions[layer_idx][0].numpy()   # (n_heads, seq, seq)
    n_heads_total = attn.shape[0]
    n_show = min(n_heads_show, n_heads_total)

    fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
    if n_show == 1:
        axes = [axes]

    for h, ax in enumerate(axes):
        im = ax.imshow(attn[h], cmap='Blues', vmin=0, vmax=attn[h].max())
        ax.set_xticks(range(len(tokens_list)))
        ax.set_xticklabels(tokens_list, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(len(tokens_list)))
        ax.set_yticklabels(tokens_list, fontsize=7)
        ax.set_title(f'Couche {layer_idx + 1} — Tête {h + 1}', fontsize=9)

    plt.suptitle(f'Poids d\'attention appris — Couche {layer_idx + 1}', fontsize=12, y=1.03)
    plt.tight_layout()
    plt.show()


# Couche 1 (basse niveau : attention syntaxique)
plot_attention_layer(attentions, tokens_list, layer_idx=0)
# Couche dernière (haut niveau : attention sémantique)
plot_attention_layer(attentions, tokens_list, layer_idx=-1)

> **Observation attendue** :  
> - Les premières couches tendent à capter des relations **locales** (mots voisins, ponctuation)  
> - Les couches profondes capturent des relations **sémantiques longue portée** (sujet ↔ verbe, coréférence)

In [ ]:
# Attention agrégée sur toutes les couches et têtes
all_attn = torch.stack(list(attentions)).mean(dim=[0, 1])[0].numpy()  # (seq, seq)

plt.figure(figsize=(10, 5))
sns.heatmap(
    all_attn,
    xticklabels=tokens_list,
    yticklabels=tokens_list,
    cmap='viridis',
    annot=True, fmt='.2f',
    linewidths=0.3
)
plt.title('Attention moyenne (toutes couches × toutes têtes)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 5.5 Cross-Attention (Encodeur → Décodeur)

Dans une architecture **encodeur-décodeur** (T5, mBART, Whisper…), le décodeur possède deux sous-couches d'attention :
1. **Self-attention causale** sur les tokens déjà générés
2. **Cross-attention** : les *Queries* viennent du décodeur, les *Keys* et *Values* viennent de l'encodeur

$$\text{CrossAttention}(Q_\text{dec},\, K_\text{enc},\, V_\text{enc}) = \text{softmax}\!\left(\frac{Q_\text{dec}\, K_\text{enc}^T}{\sqrt{d_k}}\right) V_\text{enc}$$

Contrairement au self-attention, $Q$ et $K/V$ proviennent de **séquences différentes** — d'où une matrice d'attention de taille $(n_\text{tgt} \times n_\text{src})$.

In [ ]:
# Simulation cross-attention : traduction FR → EN (sans entraînement, poids aléatoires)
source_tokens = ["l'éléphant", "rose", "a", "essayé", "de", "monter", "dans", "la", "voiture"]
target_tokens = ["The", "pink", "elephant", "tried", "to", "get", "into", "the", "car"]

n_src, n_tgt = len(source_tokens), len(target_tokens)
d_cross = 16

np.random.seed(7)
X_enc = np.random.randn(n_src, d_cross)   # sorties de l'encodeur
X_dec = np.random.randn(n_tgt, d_cross)   # états du décodeur

W_q_c = np.random.randn(d_cross, d_k) * 0.1
W_k_c = np.random.randn(d_cross, d_k) * 0.1
W_v_c = np.random.randn(d_cross, d_v) * 0.1

Q_dec = X_dec @ W_q_c   # (n_tgt, d_k)  — Queries du décodeur
K_enc = X_enc @ W_k_c   # (n_src, d_k)  — Keys   de l'encodeur
V_enc = X_enc @ W_v_c   # (n_src, d_v)  — Values de l'encodeur

# Scores cross-attention : Q_dec × K_enc^T → (n_tgt, n_src)
cross_scores  = Q_dec @ K_enc.T / np.sqrt(d_k)
cross_weights = softmax(cross_scores, axis=-1)   # (n_tgt, n_src)

plt.figure(figsize=(9, 5))
sns.heatmap(
    cross_weights, annot=True, fmt='.2f',
    xticklabels=source_tokens, yticklabels=target_tokens,
    cmap='YlOrRd', linewidths=0.5, vmin=0, vmax=cross_weights.max()
)
plt.title(
    'Cross-Attention : chaque token cible (décodeur) consulte les tokens source (encodeur)\n'
    '(poids simulés — sans entraînement)',
    fontsize=11
)
plt.xlabel('Tokens source — encodeur (Keys / Values)')
plt.ylabel('Tokens cible — décodeur (Queries)')
plt.tight_layout()
plt.show()

print(f'Q_dec  : {Q_dec.shape}  ({n_tgt} tokens décodeur, d_k={d_k})')
print(f'K_enc  : {K_enc.shape}  ({n_src} tokens encodeur, d_k={d_k})')
print(f'Matrice cross-attention : {cross_weights.shape}  → (n_tgt={n_tgt}, n_src={n_src})')
print()
print('→ Avec un vrai modèle entraîné, chaque token EN s\'alignerait sur son équivalent FR.')
print('→ Ex: "elephant" attend vers "éléphant", "car" vers "voiture".')

---
## 6. Capacités Émergentes des LLMs

On va tester les capacités **in-context learning**, raisonnement et complétion avec **NVIDIA Nemotron-Mini** — un modèle de pointe basé sur l'architecture Transformer — modèle decoder DENSE (decoder-only).

In [ ]:
from transformers import pipeline
import transformers

# Supprime les warnings verbeux de transformers
transformers.logging.set_verbosity_error()

MODEL_NAME = "nvidia/Nemotron-Mini-4B-Instruct"

# On ne fixe PAS max_new_tokens ici : on le passe à chaque appel pour éviter
# le conflit avec max_length du config du modèle.
generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    trust_remote_code=True,
)

print(f"Modèle chargé : {MODEL_NAME}")

In [ ]:
# --- Test 1 : Complétion simple (fil rouge) ---
prompt = SENTENCE

result = generator(prompt, max_new_tokens=150, do_sample=False)[0]['generated_text']
print('PROMPT  :', prompt)
print('SORTIE  :', result[:len(prompt) + 120])

In [ ]:
# --- Test 2 : In-Context Learning (Few-Shot) ---
# Le modèle apprend à faire une tâche juste via des exemples dans le prompt

few_shot_prompt = """Traduis en français :
English: The cat sleeps.
French: Le chat dort.

English: I love mathematics.
French: J'aime les mathématiques.

English: we are waiting for the bus.
French:"""

result_fs = generator(few_shot_prompt, max_new_tokens=100, do_sample=False)[0]['generated_text']
print(result_fs)

In [ ]:
# --- Test 3 : Exploration des logits (distribution sur le prochain token) ---
import torch.nn.functional as F

# Réutilisation du modèle et tokenizer déjà chargés dans generator
nem_tokenizer = generator.tokenizer
nem_model     = generator.model
nem_model.eval()

prompt_logits = "L'éléphant rose a essayé de monter dans la voiture, mais il était trop"

# Encodage et envoi sur le bon device (CPU/GPU)
inputs = nem_tokenizer(prompt_logits, return_tensors="pt").to(nem_model.device)

with torch.no_grad():
    outputs = nem_model(**inputs)
    logits = outputs.logits[0, -1, :].float()

probs = F.softmax(logits, dim=-1)

# Extraction des top tokens
top_k = 15
top_probs, top_ids = torch.topk(probs, top_k)
top_tokens = [nem_tokenizer.decode([t]).strip() for t in top_ids]

# Visualisation
semantic_ok = {"beau", "grand", "gros", "lourd","gras" ,"large", "imposant", "haut"}
colors = ["#2ecc71" if t.lower() in semantic_ok else "#3498db" for t in top_tokens]

plt.figure(figsize=(9, 4))
plt.bar(range(top_k), top_probs.cpu().numpy(), color=colors, alpha=0.85, edgecolor="white")
plt.xticks(range(top_k), top_tokens, rotation=45, ha="right", fontsize=9)
plt.ylabel("Probabilité")
plt.title(f'Top-{top_k} tokens après :\n"{prompt_logits}"', fontsize=11)
plt.grid(axis="y", alpha=0.3)

for j, tok in enumerate(top_tokens):
    if tok.lower() in semantic_ok:
        plt.text(j, top_probs[j].item() + 0.002, "*",
                 ha="center", fontsize=10, color="#27ae60", fontweight="bold")

from matplotlib.patches import Patch
plt.legend(handles=[
    Patch(color="#2ecc71", alpha=0.85, label="Token sémantiquement correct (grand, gros, lourd…)"),
    Patch(color="#3498db", alpha=0.85, label="Autre token"),
])
plt.tight_layout()
plt.show()

### Exercice 3 — Comparer temperature et top-p (nucleus) sampling

La **temperature** contrôle la "créativité" du modèle :
- $T \to 0$ : déterministe (toujours le token le plus probable)
- $T \to \infty$ : uniforme (tirage aléatoire pur)

Le **top-p** (nucleus sampling) tronque le vocabulaire aux tokens dont la probabilité cumulée dépasse $p$ :
- `top_p=0.1` : seuls les tokens couvrant les 10% premiers de la masse → très conservateur
- `top_p=1.0` : tout le vocabulaire disponible → plus diversifié

Testez les deux sur notre phrase fil rouge et observez la différence.

In [ ]:
prompt_temp = SENTENCE
print(f"PROMPT : {prompt_temp}\n")

# ── Température ──────────────────────────────────────────────────────────────
print("=" * 60)
print("  Effet de la température (top_p=1.0)")
print("=" * 60)
for temperature in [0.1, 0.7, 1.5, 3.0]:
    messages = [
        {"role": "system", "content": "Tu es un assistant. Complète la phrase en quelques mots."},
        {"role": "user",   "content": prompt_temp},
    ]
    result = generator(
        messages,
        max_new_tokens=20,
        do_sample=True,
        temperature=temperature,
        top_p=1.0,
    )[0]["generated_text"]
    answer = result[-1]["content"] if isinstance(result[-1], dict) else result
    print(f"  T={temperature:.1f} → {answer.strip()}")

# ── Top-p (nucleus sampling) ──────────────────────────────────────────────────
print()
print("=" * 60)
print("  Effet du top-p / nucleus sampling (T=1.0)")
print("=" * 60)
for top_p in [0.1, 0.5, 0.9, 1.0]:
    messages = [
        {"role": "system", "content": "Tu es un assistant. Complète la phrase en quelques mots."},
        {"role": "user",   "content": prompt_temp},
    ]
    result = generator(
        messages,
        max_new_tokens=20,
        do_sample=True,
        temperature=1.0,
        top_p=top_p,
    )[0]["generated_text"]
    answer = result[-1]["content"] if isinstance(result[-1], dict) else result
    print(f"  top_p={top_p:.1f} → {answer.strip()}")

print()
print("→ T faible / top_p faible = réponses prévisibles et répétitives.")
print("→ T élevé  / top_p élevé  = réponses créatives, parfois incohérentes.")

---
## Récapitulatif

| Composant | Rôle | Formule clé |
|---|---|---|
| **Embedding** | Discret → Vectoriel | $x_i = E[\text{id}_i]$ |
| **Positional Encoding** | Injecter l'ordre | $X_{\text{in}} = E + PE$ |
| **Self-Attention** | Mixer le contexte | $\text{softmax}(QK^T/\sqrt{d_k})V$ |
| **Causal Mask** | Empêcher la triche | $M_{ij} = -\infty$ si $i < j$ |
| **Multi-Head** | Attention parallèle | Concat$(h_1, \ldots, h_H)W_O$ |
| **FFN** | Mémoire associative | $W_2 \cdot \text{GeLU}(W_1 x)$ |
| **LayerNorm + Résidu** | Stabilité gradient | $x' = x + \text{SubBloc}(\text{LN}(x))$ |

**Pour aller plus loin :**
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — visualisations interactives
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — article original
- [BertViz](https://github.com/jessevig/bertviz) — visualisation d'attention avancée